In [ ]:
import os
from pyrogram import Client
from dotenv import load_dotenv

# Явно загружаем переменные окружения, поднявшись на уровень выше в корень проекта
load_dotenv("../.env")

api_id = os.getenv("API_ID")
api_hash = os.getenv("API_HASH")

# Создаем клиент. Файл сессии будет временно лежать прямо в папке tests
app = Client(
    name="test_session",
    api_id=int(api_id),
    api_hash=api_hash,
    workdir="."
)

print("Клиент Pyrogram готов к первому запуску.")


In [ ]:
await app.start()
print("Авторизация прошла успешно. Юзербот подключен!")

In [ ]:
import asyncio
from pyrogram import Client
from dotenv import load_dotenv

# Загружаем переменные, инициализируем клиент из папки src
app = Client(
    name="analyst_userbot",
    workdir="../src"
)

async def test_fetch_history(chat_id: int, limit: int = 5):
    messages_text = []
    async for message in app.get_chat_history(chat_id, limit=limit):
        if message.text:
            author = message.from_user.first_name if message.from_user else "Система"
            messages_text.append(f"{author}: {message.text}")
    messages_text.reverse()
    return "\n".join(messages_text)

async def run_test():
    async with app:
        me = await app.get_me()
        chat_history = await test_fetch_history(me.id, limit=5)
        print("--- Тест сбора истории из Избранного ---")
        print(chat_history)

await run_test()


In [ ]:
import os
from pyrogram import Client

# Инициализируем клиент из папки src
app = Client(
    name="analyst_userbot",
    workdir="../src"
)

async def test_download_media(chat_id: int):
    # Создаем папку для временного хранения скриншотов в корне проекта
    downloads_dir = "../src/temp_downloads"
    if not os.path.exists(downloads_dir):
        os.makedirs(downloads_dir)
        
    print("Поиск последнего медиасообщения в чате...")
    async with app:
        async for message in app.get_chat_history(chat_id, limit=20):
            # Проверяем наличие фото или картинки в документе
            if message.photo or (message.document and "image" in message.document.mime_type):
                print(f"Найдено изображение в сообщении ID {message.id}. Скачивание...")
                
                # Скачиваем файл во временную папку
                file_path = await app.download_media(
                    message, 
                    file_name=os.path.join(downloads_dir, f"screenshot_{message.id}.jpg")
                )
                print(f"Файл успешно сохранен по пути: {file_path}")
                return file_path
                
        print("Изображения в последних 20 сообщениях не найдены.")
        return None

# Убедитесь, что ваш системный VPN активен на компьютере
async with app:
    me = await app.get_me()

# Прямой асинхронный запуск теста без использования loop.run_until_complete
if me:
    await test_download_media(me.id)
